## Maize + chemistry (Maize-contrib): what’s happening?

This tutorial shows how Maize workflows become useful in cheminformatics once you start passing around **molecules** instead of plain strings.

### Key ideas

- **Maize** orchestrates computation as a *workflow graph* (nodes + connections).
- **Maize-contrib** provides chemistry-focused building blocks (under `maize.steps.mai...`) and a lightweight molecule abstraction.
- **RDKit** is used here for descriptor calculation (MW, LogP) and general chemistry utilities.

### Data types you’ll see

- **SMILES (`str`)**: compact string representation; easy to generate, but not directly dockable.
- **RDKit `Mol`**: RDKit’s in-memory molecule object.
- **`IsomerCollection` (Maize)**: a convenience wrapper around one or more *isomers/conformers* for the same compound, with safer handling for downstream workflows.

## Goal of this notebook

- Build a tiny workflow that takes a list of SMILES and computes RDKit descriptors in a **custom Maize node**.
- Get comfortable with `Input`/`Output` ports and `Return` nodes.
- Create `IsomerCollection` objects from SMILES (what Maize-contrib expects for many chemistry steps).



In [ ]:
import workshop_setup

In [ ]:
from pathlib import Path
from maize.core.workflow import Workflow
from maize.core.node import Node
from maize.steps.io import LoadData, LogResult, Void, FileParameter, Return
from maize.steps.mai.molecule import LoadSmiles, LoadMolecule, Gypsum
from maize.core.interface import Input, Output
from maize.utilities.chem import IsomerCollection
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.Draw import IPythonConsole
import pandas as pd
from pandas import DataFrame

### Explore available RDKit descriptors

RDKit ships with many built-in descriptors (TPSA, HBD/HBA, rings, etc.). The next cell prints the descriptor registry so you can see what’s available.

You **do not** need to memorize these—this is just to show that descriptor calculation is “plug-in-able” once you have an RDKit `Mol`.

In [ ]:
for desc in Descriptors._descList:
    print(desc)

**The things you should for in the cell below:**
- What the node receives, what it sends as an output
- Some properties are calculated and added in a dataframe

**The things you don't have to understand:**
- How the properties are actually calculated
- What is 'class'? What does 'self' mean? Because they are python related and out of the scope of this workshop. :)
- Basically, the things except mentioned above


In [ ]:
class CalcDesc(Node):
    out: Output[pd.DataFrame] = Output()
    inp: Input[list[str]] = Input()

    def run(self):
        res = pd.DataFrame({'smiles': self.inp.receive()})
        res['ROMol'] = res['smiles'].apply(Chem.MolFromSmiles)
        res['MolWt'] = res['ROMol'].apply(Descriptors.MolWt)
        res['MolLogP'] = res['ROMol'].apply(Descriptors.MolLogP)
        return self.out.send(res)

### Example 1: A custom node that computes descriptors

In the next cells we’ll:

- define a custom Maize node `CalcDesc`
  - **input**: `list[str]` (SMILES)
  - **output**: a `pandas.DataFrame`
- build a workflow: `LoadData → CalcDesc → Return`
- execute it and inspect the returned DataFrame

This pattern (custom node + typed ports + a small workflow) is the basic building block for integrating your own chemistry logic into Maize.

In [ ]:
mols = ['CCC', 'CCOC', 'CCCC']

In [ ]:
flow = Workflow()
load = flow.add(LoadData[list[str]])
calc = flow.add(CalcDesc)
res = flow.add(Return[pd.DataFrame])
flow.connect(load.out, calc.inp)
flow.connect(calc.out, res.inp)
load.data.set(mols)
flow.check()

In [ ]:
flow.visualize()

In [ ]:
flow.execute()

### `IsomerCollection`: Maize’s molecule container

Many Maize-contrib chemistry nodes operate on **`IsomerCollection`** rather than raw SMILES.

Why?

- One “compound” can have **multiple isomers** (tautomers, stereoisomers) and **multiple conformers**.
- Downstream steps (like docking) often need that richer representation.

In the next cell we create an `IsomerCollection` from a SMILES string and inspect what it contains.

In [ ]:
res.get()

In [ ]:
from maize.utilities.chem.chem import IsomerCollection
iso1 = IsomerCollection.from_smiles('CCC(O)CN')

In [ ]:
iso1

In [ ]:
iso1.molecules[0]._molecule

In [ ]:
iso1.molecules[1]._molecule

### Example 2: Converting SMILES to `IsomerCollection` in a workflow

Here we create another tiny workflow:

`LoadData (SMILES list) → Smi2Mols → Return`

so you can see how to pass “molecule objects” through Maize nodes (instead of only passing strings).

This is the kind of conversion step you typically place right after any SMILES generator (including REINVENT) before you call conformer generation, docking, or scoring nodes.

In [ ]:
class Smi2Mols(Node):
    inp: Input[list[str]] = Input()
    out: Output[list[IsomerCollection]] = Output()

    def run(self) -> None:
        smiles_list = self.inp.receive()
        mols = [IsomerCollection.from_smiles(smi) for smi in smiles_list]
        self.out.send(mols)

In [ ]:
flow = Workflow()
load = flow.add(LoadData[list[str]])
emb = flow.add(Smi2Mols)
ret = flow.add(Return[list[IsomerCollection]])
flow.connect(load.out, emb.inp)
flow.connect(emb.out, ret.inp)
load.data.set(['CC', 'CCC(O)CN'])
flow.check()

In [ ]:
flow.execute()

In [ ]:
results = ret.get()

In [ ]:
results